In [11]:
from nltk.corpus import stopwords
import re
import os
from typing import List, Dict
import numpy as np
import pandas as pd
from charset_normalizer import from_path
from collections import Counter
import nltk


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [2]:
def tokenize(text):
    #pattens for respectively numbers, words and punctuation 
    token_pattern = r""" 
        \w+(?:[-'’]\w+)?
    """
    text = text.lower() #transforme all the text to lowercase
    pattern = re.compile(token_pattern, re.VERBOSE) #compile the regular expressions
    
    tokens = pattern.findall(text)  #extract all tokens from the input text that matches the pattern
    
    stop_words = set(stopwords.words('english'))
    filtered_sentence = []
     
    for w in tokens:
        if w not in stop_words:
            filtered_sentence.append(w)
     
    print("tokens",tokens)
    print("\n")
    print("tokens without stopwords",filtered_sentence)
    print("and" in filtered_sentence)
    return tokens

In [4]:
print(stopwords.words('english'))


NameError: name 'stopwords' is not defined

In [5]:
def read_file(file_path):
    """
    Read the file with the detected encoding.
    """
    #TODO: fix CharsetMatch problem
    result = from_path(file_path).best() #detects best encoding for the file
    if result:
        return str(result)  
    else:
        raise ValueError(f"Could not detect encoding for file: {file_path}")


In [6]:
class CountVectorFeaturizer:
    def __init__(self):
        self.vocabulary = {}  

    def fit(self, documents: List[str]) -> None:
        
        token_counter = Counter() #using a counter to keep track of frequencies 
        self.tokenized_docs = []
        
        # Tokenize all docs
        self.tokenized_docs = [tokenize(doc) for doc in documents]
        for tokens in self.tokenized_docs:
            token_counter.update(tokens) #extract unique tokens
        
        # Map each token to a unique index based on frequency
        self.vocabulary = {token: idx for idx, (token, _) in enumerate(token_counter.most_common())}

    def transform(self, documents: List[str]) -> np.ndarray:
        
        # Initialize a matrix for count vectors Featurizer
        matrix = np.zeros((len(documents), len(self.vocabulary)), dtype=int)

        #counting freq of each token in the vocab
        for i, tokens in enumerate(self.tokenized_docs):
            for token in tokens:
                idx = self.vocabulary.get(token)
                if idx is not None:
                    matrix[i, idx] += 1
        
        return matrix


In [8]:
folder_path = "../data-plagiarism"
documents = []

featurizer = CountVectorFeaturizer()
for file_name in os.listdir(folder_path):
    if file_name.endswith(".txt"): 
        print("file name", file_name)
        file_path = os.path.join(folder_path, file_name)
        content = read_file(file_path)
        documents.append(content)

featurizer.fit(documents)
count_matrix = featurizer.transform(documents)

#print("Vocabulary:", featurizer.vocabulary)
#print("Count Matrix:\n", count_matrix)
#print("\n") 


file name g0pA_taska.txt
file name g0pA_taskb.txt
file name g0pA_taskc.txt
file name g0pA_taskd.txt
file name g0pA_taske.txt
file name g0pB_taska.txt
file name g0pB_taskb.txt
file name g0pB_taskc.txt
file name g0pB_taskd.txt
file name g0pB_taske.txt
file name g0pC_taska.txt
file name g0pC_taskb.txt
file name g0pC_taskc.txt
file name g0pC_taskd.txt
file name g0pC_taske.txt
file name g0pD_taska.txt
file name g0pD_taskb.txt
file name g0pD_taskc.txt
file name g0pD_taskd.txt
file name g0pD_taske.txt
file name g0pE_taska.txt
file name g0pE_taskb.txt
file name g0pE_taskc.txt
file name g0pE_taskd.txt
file name g0pE_taske.txt
file name g1pA_taska.txt
file name g1pA_taskb.txt
file name g1pA_taskc.txt
file name g1pA_taskd.txt
file name g1pA_taske.txt
file name g1pB_taska.txt
file name g1pB_taskb.txt
file name g1pB_taskc.txt
file name g1pB_taskd.txt
file name g1pB_taske.txt
file name g1pD_taska.txt
file name g1pD_taskb.txt
file name g1pD_taskc.txt
file name g1pD_taskd.txt
file name g1pD_taske.txt


NameError: name 'stopwords' is not defined

In [9]:
expected_results = pd.read_csv(os.path.join(folder_path, "test_info.csv"), sep=",")
#print(expected_results)

#extracting labels that we'll be using in our classif
labels = dict(zip(expected_results["File"], expected_results['Category']))
print(labels)

#combine heavy and cut as plagiat, light orig as non_plagiat
def map_value(value):
    return 'plagiat' if value in ['heavy', 'cut'] else 'non_plagiat'

filtered_labels = { k: map_value(v) for k, v in sorted(labels.items()) } #filters the files

total_files = len(filtered_labels)
print("total_files", total_files)

plagiat_files = sum(1 for v in filtered_labels.values() if v == "plagiat")
print("plagiat_files", plagiat_files)

probability_plagiat = plagiat_files/total_files
print("probability_plagiat", probability_plagiat)

print("probability_plagiat", 1-probability_plagiat)

print("filtered_labels", filtered_labels)

{'g0pB_taske.txt': 'heavy', 'g0pC_taska.txt': 'heavy', 'g0pC_taskb.txt': 'non', 'g0pC_taskc.txt': 'non', 'g0pC_taskd.txt': 'cut', 'g0pC_taske.txt': 'light', 'g1pB_taskb.txt': 'non', 'g1pB_taskc.txt': 'heavy', 'g1pB_taskd.txt': 'light', 'g1pB_taske.txt': 'cut', 'g1pD_taska.txt': 'light', 'g1pD_taskb.txt': 'cut', 'g3pB_taskd.txt': 'light', 'g3pB_taske.txt': 'cut', 'g3pC_taska.txt': 'cut', 'g3pC_taskb.txt': 'non', 'g3pC_taskc.txt': 'non', 'g3pC_taskd.txt': 'heavy', 'g3pC_taske.txt': 'light', 'g4pB_taska.txt': 'non', 'g4pB_taskb.txt': 'non', 'g4pB_taskc.txt': 'heavy', 'g4pB_taskd.txt': 'light', 'g4pB_taske.txt': 'cut', 'g4pC_taska.txt': 'cut', 'g4pC_taskb.txt': 'non', 'g4pC_taskc.txt': 'non', 'g4pC_taskd.txt': 'heavy', 'g4pC_taske.txt': 'light', 'g4pD_taska.txt': 'light', 'g4pD_taskb.txt': 'cut', 'g4pD_taskc.txt': 'non', 'g4pD_taskd.txt': 'non', 'g4pD_taske.txt': 'heavy', 'g4pE_taska.txt': 'heavy', 'g4pE_taskb.txt': 'light', 'g4pE_taske.txt': 'non', 'orig_taska.txt': 'orig', 'orig_taskb.tx

In [12]:
docs = ["chat noir, chat blanc"]
featurizer.fit(docs)
print(featurizer.transform(docs))

NameError: name 'stopwords' is not defined